# Sentiment Analysis Pipeline — Optimizing Inference with Batch Processing

In [2]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from scipy.special import softmax
import urllib.request
import numpy as np
import pandas as pd
import torch
import time
import csv
import re

/home/nhx/miniconda/envs/ai_app/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import warnings
warnings.filterwarnings('ignore')

## 2. Identifying the Performance Bottleneck

In the previous implementation, the model was applied sequentially  
to each input.

When processing multiple comments, this resulted in:
- Increased execution time
- Inefficient use of the model

This behavior highlights a performance bottleneck in the pipeline.

In [4]:
def preprocess(text):
    new_text = []
    for t in text.split(" "):
        t = "@user" if t.startswith("@") and len(t) > 1 else t
        t = "http" if t.startswith("http") else t
        new_text.append(t)
    return " ".join(new_text)

In [5]:
task = "sentiment"
MODEL = f"cardiffnlp/twitter-roberta-base-{task}"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL)

labels = []
mapping_link = f"https://raw.githubusercontent.com/cardiffnlp/tweeteval/main/datasets/{task}/mapping.txt"

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 9428.46it/s]


In [6]:
with urllib.request.urlopen(mapping_link) as f:
    html = f.read().decode("utf-8").split("\n")
    csvreader = csv.reader(html, delimiter="\t")
    labels = [row[1] for row in csvreader if len(row) > 1]

In [7]:
def split_into_phrases(text):
    parts = re.split(r"\s*(?:\.|\bbut\b|\bhowever\b)\s*",text,flags=re.IGNORECASE)
    return [p.strip() for p in parts if p.strip()]

In [8]:
ASPECTS = ["design", "performance", "price", "service", "quality", "delivery", "staff"]

In [9]:
def get_aspects(phrase):
    found_aspects = []
    for aspect in ASPECTS:
        if aspect.lower() in phrase.lower():
            found_aspects.append(aspect.lower())
    return found_aspects

## 3. Implementing Batch Processing

We introduce a batch prediction function that:

- Takes a list of phrases as input
- Processes them in a single model call
- Returns predictions for all inputs at once

This reduces the number of calls to the model  
and improves overall efficiency.

In [10]:
model.eval()
def predict_batch(texts):
    texts = [preprocess(t) for t in texts]
    encoded_input = tokenizer(texts,return_tensors="pt",padding=True,truncation=True,max_length=128)

    with torch.no_grad():
        output = model(**encoded_input)

    scores = output.logits.detach().numpy()
    scores = softmax(scores, axis=1)
    results = []

    for score in scores:
        ranking = np.argsort(score)[::-1]
        label = labels[ranking[0]]
        confidence = round(float(score[ranking[0]]), 2)

        results.append({"Label": label,"Score": confidence})

    return results

In [11]:
def analyze_comment_batch(text):
    results_with_aspects = []
    results_without_aspects = []

    all_phrases = []

    for t in text:
        phrase = split_into_phrases(t)
        all_phrases.extend(phrase)

    predictions = predict_batch(all_phrases)

    for phrase, pred in zip(all_phrases, predictions):
        aspects = get_aspects(phrase)

        if aspects:
            results_with_aspects.append({"Phrase": phrase,"Aspects": aspects,"Label": pred["Label"],"Score": pred["Score"]})
        else:
            results_without_aspects.append({"Phrase": phrase,"Label": pred["Label"],"Score": pred["Score"]})

    return {
        "with_aspects": results_with_aspects,
        "without_aspects": results_without_aspects
    }

## 4. Testing the Optimized Pipeline

We test the updated pipeline on multiple comments.

The results are compared with the previous implementation to ensure:
- Predictions remain consistent
- Output structure is preserved
- Performance is improved

In [12]:
test_comments = [
    # Straightforward + known aspect
    "The design is beautiful.",
    "The price is too expensive.",
    
    # Straightforward + known aspect + double aspect
    "The staff and service are nice.",

    # Straightforward + no known aspect
    "I really loved it.",
    "This was a complete waste of time.",

    # Positive then negative, both with known aspects
    "The design is beautiful but the performance is terrible.",
    "The staff were friendly but the service was very slow.",

    # Negative then positive, both with known aspects
    "The price is high but the quality is excellent.",
    "The performance was bad at first but the service was great.",

    # Complex: one part has aspect, one part has no aspect
    "The design is amazing but I still regret buying it.",
    "I hated the experience at first but the staff were very kind.",

    # Multiple aspects in one sentence part
    "The design and performance are both excellent.",
    "The price and delivery were disappointing.",

    # No aspect at all, mixed sentiment
    "I liked it at first but it became disappointing later.",
    "At the beginning it was confusing but in the end it was useful.",

    # Edge cases
    "The product is okay.",
    "Not bad, but not amazing either.",
    "The service was not terrible, but it was not great.",
    "The delivery was fast however the package looked damaged.",
]

In [13]:
start_time = time.perf_counter()
result = analyze_comment_batch(test_comments)
end_time = time.perf_counter()

print(f"This batch function took {end_time - start_time} s to be executed")

This batch function took 6.312290622037835 s to be executed


In [14]:
df_aspects = pd.DataFrame(result["with_aspects"])
df_aspects.head(10)

,Phrase,Aspects,Label,Score
0,The design is beautiful,[design],positive,0.98
1,The price is too expensive,[price],negative,0.87
2,The staff and service are nice,"[service, staff]",positive,0.98
3,The design is beautiful,[design],positive,0.98
4,the performance is terrible,[performance],negative,0.97
5,The staff were friendly,[staff],positive,0.90
6,the service was very slow,[service],negative,0.94
7,The price is high,[price],neutral,0.60
8,the quality is excellent,[quality],positive,0.96
9,The performance was bad at first,[performance],negative,0.95


In [15]:
df_w_aspects = pd.DataFrame(result["without_aspects"])
df_w_aspects.head(5)

,Phrase,Label,Score
0,I really loved it,positive,0.98
1,This was a complete waste of time,negative,0.98
2,I still regret buying it,negative,0.89
3,I hated the experience at first,negative,0.97
4,I liked it at first,positive,0.91


### Conclusion

The optimized pipeline significantly improves performance  
by introducing batch processing.

Key improvements:
- Faster inference
- Better scalability
- More efficient model usage

The system is now closer to a production-ready solution  
and can be integrated into real-time applications such as APIs.